## Enriching stock market data using Open AI API 

<p align="center">
    <img src="images/nasdaq100.png" width="450">
</p>

The Nasdaq-100 is a stock market index made up of 101 equity securities issued by 100 of the largest non-financial companies listed on the Nasdaq stock exchange. It helps investors compare stock prices with previous prices to determine market performance.

In this project you are provided with two CSV files containing Nasdaq-100 stock information:
- _**nasdaq100_CA.csv**_: contains information about companies in the index such as symbol, name, etc. For this analysis, only companies headquartered in California have been selected.
- _**nasdaq100_price_change.csv**_: contains price changes per stock across periods including (but not limited to) one day, five days, one month, six months, one year, etc.

As an AI developer, you will leverage the OpenAI API to classify companies into sectors and produce a summary of sector and company performance for this year, for the companies in the index that are headquartered in California.

# CSV with Nasdaq-100 stock data

In this project, you have available two CSV files `nasdaq100_CA.csv` and `nasdaq100_price_change.csv`.

## nasdaq100_CA.csv

```py
symbol,name,headQuarter,dateFirstAdded,cik,founded
AAPL,Apple Inc.,"Cupertino, CA",,0000320193,1976-04-01
ABNB,Airbnb,"San Francisco, CA",,0001559720,2008-08-01
ADBE,Adobe Inc.,"San Jose, CA",,0000796343,1982-12-01
...
```

## nasdaq100_price_change.csv

```py
symbol,1D,5D,1M,3M,6M,ytd,1Y,3Y,5Y,10Y,max
AAPL,-1.7254,-8.30086,-6.20411,3.042,15.64824,42.99992,8.47941,60.96299,245.42031,976.99441,139245.53954
ABNB,2.1617,-2.21919,9.88336,19.43286,19.64241,68.66902,23.64013,-1.04347,-1.04347,-1.04347,-1.04347
ADBE,0.5409,-1.77817,9.16191,52.0465,38.01522,57.22723,21.96206,17.83037,109.05718,1024.69214,251030.66399
ADI,0.9291,-4.03352,2.58486,3.65887,5.01602,17.02062,8.09735,63.42847,92.81874,286.77518,26012.63736
...
```

In [2]:
# Start your code here!
import os
import pandas as pd
from openai import OpenAI

# Instantiate an API client
client = OpenAI()

# ---------------------------------------------------------------
# Step 1: Build nasdaq100_ca with a "ytd" column
# ---------------------------------------------------------------
nasdaq100_ca = pd.read_csv("nasdaq100_CA.csv")
price_change = pd.read_csv("nasdaq100_price_change.csv")

# Merge the YTD performance column in on the symbol
nasdaq100_ca = nasdaq100_ca.merge(
    price_change[["symbol", "ytd"]],
    on="symbol",
    how="left",
)

print(nasdaq100_ca.head())
# ---------------------------------------------------------------
# Step 2: Use the OpenAI API to classify each stock into a sector
# ---------------------------------------------------------------
ALLOWED_SECTORS = [
    "Technology", "Consumer Cyclical", "Industrials", "Utilities",
    "Healthcare", "Communication", "Energy", "Consumer Defensive",
    "Real Estate", "Financial",
]

def classify_sector(symbol: str, name: str) -> str:
    """Ask the model to classify a single company into one of the allowed sectors."""
    prompt = (
        f"Classify the company '{name}' (ticker: {symbol}) into exactly ONE "
        f"of the following sectors. Respond with ONLY the sector name, no "
        f"punctuation or extra text.\n\n"
        f"Allowed sectors: {', '.join(ALLOWED_SECTORS)}"
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a financial analyst that classifies companies into sectors."},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
        max_tokens=10,
    )

    sector = response.choices[0].message.content.strip()
    # Defensive: snap to an allowed value if the model deviates slightly
    for allowed in ALLOWED_SECTORS:
        if allowed.lower() in sector.lower():
            return allowed
    return sector

# Classify every company in the California subset
nasdaq100_ca["sector"] = nasdaq100_ca.apply(
    lambda row: classify_sector(row["symbol"], row["name"]),
    axis=1,
)

print(nasdaq100_ca[["symbol", "name", "ytd", "sector"]].head())
# ---------------------------------------------------------------
# Step 3: Use the OpenAI API to recommend the 2 best sectors
# and 2+ companies per sector, based on YTD performance
# ---------------------------------------------------------------

# Build a compact, per-sector summary the model can reason over
sector_summary = (
    nasdaq100_ca
    .groupby("sector")
    .agg(
        avg_ytd=("ytd", "mean"),
        num_companies=("symbol", "count"),
        companies=("name", lambda s: ", ".join(s)),
        ytd_by_company=(
            "ytd",
            lambda s: ", ".join(
                f"{n} ({y:.2f}%)"
                for n, y in zip(
                    nasdaq100_ca.loc[s.index, "name"], s
                )
            ),
        ),
    )
    .sort_values("avg_ytd", ascending=False)
    .reset_index()
)

summary_text = "\n".join(
    f"- {row.sector}: avg YTD {row.avg_ytd:.2f}% across {row.num_companies} "
    f"company/companies — {row.ytd_by_company}"
    for row in sector_summary.itertuples()
)

recommendation_prompt = f"""
You are a financial analyst. Below is year-to-date (YTD) performance data for
California-headquartered Nasdaq-100 companies, grouped by sector.

{summary_text}

Based on this data:
1. Identify the TWO best-performing sectors (by average YTD return).
2. For EACH of those two sectors, recommend two or more specific companies
   from the list above as the most attractive picks, and briefly explain why
   (mention their YTD performance).
3. Conclude with a short overall summary.

Format the response in clear, readable prose with section headings for each
recommended sector.
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a knowledgeable financial analyst providing concise, data-driven stock recommendations."},
        {"role": "user", "content": recommendation_prompt},
    ],
    temperature=0.3,
    max_tokens=600,
)

stock_recommendations = response.choices[0].message.content
print(stock_recommendations)

  symbol               name        headQuarter  ...      cik     founded       ytd
0   AAPL         Apple Inc.      Cupertino, CA  ...   320193  1976-04-01  42.99992
1   ABNB             Airbnb  San Francisco, CA  ...  1559720  2008-08-01  68.66902
2   ADBE         Adobe Inc.       San Jose, CA  ...   796343  1982-12-01  57.22723
3   ADSK           Autodesk     San Rafael, CA  ...   769397  1982-01-30  10.02701
4   AMAT  Applied Materials    Santa Clara, CA  ...     6951  1967-11-10  55.46366

[5 rows x 7 columns]
  symbol               name       ytd             sector
0   AAPL         Apple Inc.  42.99992         Technology
1   ABNB             Airbnb  68.66902  Consumer Cyclical
2   ADBE         Adobe Inc.  57.22723         Technology
3   ADSK           Autodesk  10.02701         Technology
4   AMAT  Applied Materials  55.46366         Technology
